# Episode 4 — 7-segment display and buzzer

Both blocks do their timing in the PL, so Python only writes a register now and then:

* **7-segment display:** the four digits share segment lines. Logic in the PL lights one digit
  at a time, switching about 1500 times a second, so all four look steady. Python just writes which
  characters to show.
* **Buzzer:** logic in the PL toggles the buzzer pin every *N* clock cycles of the 100 MHz PL clock.
  Python writes *N*; the note plays until you turn it off.

Fit the jumper on **J10 pins 1–2** to connect the buzzer.

In [ ]:
from pynq.overlays.base import BaseOverlay
import time

base = BaseOverlay("base.bit")
seg = base.seven_seg
bz = base.buzzer

## 1. Numbers, hex and text

In [ ]:
seg.show_number(1234)
time.sleep(1)
seg.show_number(3.14159)      # as many decimals as fit
time.sleep(1)
seg.show_hex(0xC0DE)
time.sleep(1)
seg.show_text("HI")           # right aligned
time.sleep(1)
seg.show_text("Good")

## 2. Your own segment patterns

Each digit is 8 bits: bit 0 = segment a … bit 6 = segment g, bit 7 = decimal point.

```
  -a-
 f   b
  -g-
 e   c
  -d-  .dp
```

In [ ]:
spinner = [0x01, 0x02, 0x04, 0x08, 0x10, 0x20]   # a, b, c, d, e, f
for i in range(24):
    p = spinner[i % 6]
    seg.show_segments([p, p, p, p])
    time.sleep(0.08)
seg.clear()

## 3. A stopwatch

* **Button 0** starts and stops.
* **Button 1** resets (when stopped).
* Flip **switch 0** to end the cell (it also ends after 60 seconds).

In [ ]:
running = False
elapsed = 0.0
started = 0.0
shown = 0.0
last_buttons = 0
first_switch = base.switches[0].read()
end = time.time() + 60

while time.time() < end and base.switches[0].read() == first_switch:
    buttons = base.buttons.read()
    pressed = buttons & ~last_buttons          # buttons that just went down
    last_buttons = buttons
    if pressed & 0b01:                         # button 0: start / stop
        if running:
            elapsed += time.time() - started
        else:
            started = time.time()
        running = not running
    if pressed & 0b10 and not running:         # button 1: reset
        elapsed = 0.0
    shown = elapsed + (time.time() - started if running else 0.0)
    seg.show_number(shown, decimals=2 if shown < 100 else 1)
    time.sleep(0.02)

print("Final time: {:.2f} s".format(shown))

## 4. Beeps and notes

The buzzer frequency is set by the half period in clock cycles:
`half_period = 100 MHz / (2 × frequency)`.

In [ ]:
bz.beep(1000, 0.2)
time.sleep(0.2)
bz.beep(2000, 0.2)

## 5. A tune from note names

A note's frequency doubles every octave (12 semitones), starting from A4 = 440 Hz.
The note name is shown on the display while it plays.

In [ ]:
SEMITONE = {"C": -9, "D": -7, "E": -5, "F": -4, "G": -2, "A": 0, "B": 2}

def note_hz(name):
    """'A4' -> 440.0 Hz, 'C5' -> 523.3 Hz"""
    letter, octave = name[0], int(name[1])
    return 440.0 * 2 ** ((SEMITONE[letter] + 12 * (octave - 4)) / 12)

# Ode to Joy (Beethoven, 1824): (note, beats)
ode_to_joy = [
    ("E4", 1), ("E4", 1), ("F4", 1), ("G4", 1), ("G4", 1), ("F4", 1), ("E4", 1), ("D4", 1),
    ("C4", 1), ("C4", 1), ("D4", 1), ("E4", 1), ("E4", 1.5), ("D4", 0.5), ("D4", 2),
    ("E4", 1), ("E4", 1), ("F4", 1), ("G4", 1), ("G4", 1), ("F4", 1), ("E4", 1), ("D4", 1),
    ("C4", 1), ("C4", 1), ("D4", 1), ("E4", 1), ("D4", 1.5), ("C4", 0.5), ("C4", 2),
]

BEAT = 0.35   # seconds
for name, beats in ode_to_joy:
    seg.show_text(name)
    bz.tone(note_hz(name))
    time.sleep(beats * BEAT * 0.9)
    bz.off()
    time.sleep(beats * BEAT * 0.1)
seg.clear()

## Try it yourself

Make a siren: sweep the buzzer from 500 Hz up to 1500 Hz and back down, three times.
Hint: `bz.tone(f)` in a loop with a short `time.sleep()`, then `bz.off()`.

In [ ]:
# your code here
bz.off()